# Модуль с методом k ближайших соседей

In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv("data\\train.csv", index_col='id')

Применяем те же преобразования, что и в show_data.ipynb и исключим признаки, которые имеют много категориальных значений:

In [6]:
from sklearn.preprocessing import FunctionTransformer

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.fillna({'Profession': 'unemployed'}, inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

Применяем OneHot энкодинг для категориальных признаков и StandardScaler для стандартизации

In [12]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector
import numpy as np

encoder = ColumnTransformer([
    ('scaler', StandardScaler(), make_column_selector(dtype_include=np.number)),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), make_column_selector(dtype_include=['str']))
])

Строим пайплайн

In [32]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('prep', prep),
    ('encoder', encoder),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

In [33]:
X = df_train.iloc[:, :-1]
y = df_train.iloc[:, -1]

Оценивать качество будем при помощи кросс-валидации

In [34]:
from sklearn.model_selection import cross_val_score

def cv(model, X, y):
    scores = cross_val_score(
        estimator=model,
        X=X,
        y=y,
        scoring='f1',
        cv=5
    )

    return scores.mean()

In [35]:
cv(pipe, X, y)

np.float64(0.791533652595128)

In [36]:
pipe.fit(X, y)
X_test = pd.read_csv("data\\test.csv", index_col='id')
y_pred = pipe.predict(X_test)
df_pred = pd.DataFrame(y_pred, columns=['Depression'])
df_pred.index = df_pred.index + 1
df_pred.to_csv("out\\knn.csv", index_label='id')